# Quick start

This notebook converts the IEEE 13-node feeder from GridLAB-D to CIM and back. It runs from `docs/guide/` in the
repository, which holds the model at `../../tests/decks/IEEE/IEEE13_cdpsm.glm`. Install the package first
([Installation](installation.md)).

| Step | Call | Touches files |
|---|---|---|
| Read | `GlmReader(path, defines=...).read_all()` returns a `GlmModel` | reads the `.glm`, its `#include`s and its player files |
| Import | `GlmImporter(name).to_cim(model)` returns a cim18v17 network | writes `.cimtbl` artifacts to a temporary directory unless you pass `artifact_dir=` |
| Export | `GlmExporter().to_model(network)` returns a `GlmModel` | none |
| Write | `GlmWriter().write(model, path)` | writes the `.glm` |

Only the reader and the writer read and write model files. `to_cim` writes its `.cimtbl` text to disk only to build the network from it.

## Read the model

`GlmReader` follows `#include`s, expands `${NAME}` macros and types every value. This model uses `${VSOURCE}`, the
source voltage that `gridlabd -D VSOURCE=...` would pass, so it goes in `defines`. `read_all` returns a `GlmModel`:
records keyed by their GLM class.

In [1]:
from pathlib import Path

from cimhub_gridlabd.glm_reader import GlmReader

deck = Path("../../tests/decks/IEEE/IEEE13_cdpsm.glm")
model = GlmReader(deck, defines={"VSOURCE": "66395.28"}).read_all()
for glm_class, records in model.graph.items():
    print(f"{glm_class.class_name:28} {len(records)}")

regulator_configuration      1
regulator                    1
line_configuration           7
transformer_configuration    3
capacitor                    2
inverter                     5
solar                        2
battery                      3
overhead_line                11
switch                       3
fuse                         1
recloser                     1
transformer                  3
substation                   1
node                         19
triplex_node                 1
triplex_meter                2
triplex_load                 1
load                         9
meter                        2


Each record is a dataclass from the LinkML schema. Slots keep the units the `.glm` used. This load's `current_pf_C`
is a power factor and its `nominal_voltage` is in volts.

In [2]:
from cimhub_gridlabd.schema import gridlabd as schema

print(model.list_by_class(schema.Load)[0])

Load({
  'name': 'ld_611',
  'parent': '611',
  'phases': 'CN',
  'nominal_voltage': 2401.78,
  'current_pf_C': 0.904819,
  'current_fraction_C': 1.0
})


## Import to CIM

`GlmImporter(name).to_cim` maps the records to `.cimtbl` text and has cimtables build the CIM network from it. `name`
is the master document's `Model` and file stem. `artifact_dir` keeps the text; here it goes to a temporary directory.
(`glm_to_cim(model, name=...)` is the same call as one function.)

The importer logs one warning here. The models exported by the GridAPPS-D CIMHub Java
tools drive every load by a player named `ieeezipload`, which the run file that includes the model defines, not the
model. The warning says that name is in no player of the deck, so the loads' group is written without a schedule.

In [3]:
import tempfile

from cimhub_gridlabd.importer.glm_to_cimtbl import GlmImporter

work = Path(tempfile.mkdtemp())
network = GlmImporter("IEEE13_cdpsm").to_cim(model, artifact_dir=work / "cimtbl")

1 player(s) named by loads are not in the deck: ['ieeezipload']; their groups have no schedule


The network is a cim-graph model. `network.graph` holds its objects by CIM class. The model's 11 `overhead_line`s
are 11 `ACLineSegment`s, its three `switch`es, one `fuse` and one `recloser` are switch classes of their own, and its
three `transformer`s are `PowerTransformer`s.

In [4]:
import cimprofiles.cim18.cim18v17 as cim

for cim_class in (cim.ConnectivityNode, cim.ACLineSegment, cim.LoadBreakSwitch, cim.Fuse, cim.Recloser,
                  cim.PowerTransformer, cim.VoltageRegulator, cim.ConformLoad, cim.LinearShuntCompensator,
                  cim.PowerElectronicsConnection):
    print(f"{cim_class.__name__:28} {len(network.graph.get(cim_class, {}))}")

ConnectivityNode             21
ACLineSegment                11
LoadBreakSwitch              3
Fuse                         1
Recloser                     1
PowerTransformer             3
VoltageRegulator             3
ConformLoad                  10
LinearShuntCompensator       2
PowerElectronicsConnection   5


To read a value, use a [CIMHub core getter](https://pnnl-cim-tools.github.io/CIMHub/core/guide/getters/). It converts
CIM's SI value to the unit you name. Load 611 is a 170 kW, 80 kVAr load on phase C.

In [5]:
from cimhub_core.getters.load import get_phase_loads

load = next(obj for obj in network.graph[cim.ConformLoad].values() if obj.name == "ld_611")
loads = get_phase_loads(load, "kW", "kVAr")
print(loads.total)
print([phase.name for phase in loads.phases])

PhasePower(p=170.00005388786002, q=79.99988013221203)
['C']


## The `.cimtbl` artifacts

`artifact_dir` holds one file per GLM class the model uses, plus the master, which `Import`s the others in dependency
order (catalogs and configurations before the equipment that uses them).

In [6]:
artifacts = work / "cimtbl"
print(sorted(path.name for path in artifacts.iterdir()))
print()
imports = [line for line in (artifacts / "IEEE13_cdpsm.cimtbl").read_text().splitlines() if line.startswith("Import")]
print("\n".join(imports))

['IEEE13_cdpsm.cimtbl', 'bus.cimtbl', 'capacitor.cimtbl', 'fuse.cimtbl', 'inverter.cimtbl', 'line_configuration.cimtbl', 'load.cimtbl', 'overhead_line.cimtbl', 'player.cimtbl', 'recloser.cimtbl', 'regulator.cimtbl', 'source.cimtbl', 'switch.cimtbl', 'transformer.cimtbl', 'transformer_configuration.cimtbl', 'triplex_load.cimtbl', 'zip_fractions.cimtbl']

Import bus.cimtbl
Import source.cimtbl
Import zip_fractions.cimtbl
Import line_configuration.cimtbl
Import transformer_configuration.cimtbl
Import overhead_line.cimtbl
Import switch.cimtbl
Import fuse.cimtbl
Import recloser.cimtbl
Import transformer.cimtbl
Import regulator.cimtbl
Import player.cimtbl
Import load.cimtbl
Import triplex_load.cimtbl
Import capacitor.cimtbl
Import inverter.cimtbl


Values are copied as the `.glm` gave them, and the unit is in the column header (`p (W)`). A comment says what the
importer defaulted or left out.

In [7]:
lines = (artifacts / "load.cimtbl").read_text().splitlines()
print("\n".join(line[:130] for line in lines[9:14]))

Table ConformLoad: mRID, name, node, phases, phaseConnection, grounded, p (W), q (VAr), customerCount, LoadResponse, BaseVoltage, 
    ld_ld_611,  ld_611,  611, C,   Y, true,  170000.05388786,    79999.88013221203,  1, zip_constant_i, bv_4p16kv, lg_ieeezipload
    ld_ld_634c, ld_634c, 634, ABC, Y, true,  400000.01604496,    289999.979504666,   1, zip_constant_i, bv_480v,   lg_ieeezipload
    ld_ld_645,  ld_645,  645, B,   Y, true,  169999.99857148,    125000.00454226874, 1, zip_constant_i, bv_4p16kv, lg_ieeezipload
    ld_ld_646,  ld_646,  646, BC,  D, false, 229999.89835648998, 132000.18380327037, 1, zip_constant_i, bv_4p16kv, lg_ieeezipload


The master's comment lists every GLM class in the model that no mapper reads, as `not mapped: <class> <count>`. This
model has none, so the comment names only the mapping spec. Its `BaseFrequency` row says the frequency was defaulted,
because the model sets no `nominal_frequency`.

In [8]:
print("\n".join((artifacts / "IEEE13_cdpsm.cimtbl").read_text().splitlines()[:13]))

# GridLAB-D model IEEE13_cdpsm
# Mapping: cimhub_gridlabd/docs/CIMTBL_MAPPING.md#master

Profile = cim18v17
Authority = https://github.com/PNNL-CIM-Tools/CIMHub
Model = IEEE13_cdpsm

Object Substation: mRID=sub_IEEE13_cdpsm, name=IEEE13_cdpsm

Object Feeder: mRID=fdr_IEEE13_cdpsm, name=IEEE13_cdpsm, NormalEnergizingSubstation=sub_IEEE13_cdpsm

# defaulted: powerflow::nominal_frequency=60.0
Object BaseFrequency: mRID=bf_60hz, frequency (Hz)=60.0


## Export to a `.glm`

`GlmExporter().to_model` reads the CIM network through the core getters and returns a `GlmModel`, the same shape the
reader returns. `name_source="name"` writes each GLM name from the CIM `name`; the default is the `mRID`.

In [9]:
from cimhub_gridlabd.exporter.cim_to_glm import GlmExporter

exported = GlmExporter().to_model(network, name_source="name")
for glm_class, records in exported.graph.items():
    print(f"{glm_class.class_name:28} {len(records)}")

1 load group(s) are written static (no schedule, no points, or a NonConformLoad's): ['ieeezipload']


substation                   1
node                         19
triplex_node                 1
line_configuration           7
overhead_line                11
switch                       3
fuse                         1
recloser                     1
transformer_configuration    3
transformer                  3
regulator_configuration      1
regulator                    1
load                         9
triplex_load                 1
capacitor                    2
inverter                     5
battery                      3
solar                        2


The exporter logs the same fact once more: the load group `ieeezipload` has no schedule, so its loads are exported
at their base power. It writes the header of one power flow: a `clock` that starts and stops at one instant, the Newton-Raphson
solver at the network's base frequency, and `line_capacitance`. `GlmWriter().write` creates the `.glm` and its folders.

In [10]:
from cimhub_gridlabd.glm_writer import GlmWriter

written = work / "out" / "IEEE13_cdpsm.glm"
GlmWriter().write(exported, written)
print(sorted(path.name for path in written.parent.iterdir()))
print()
print("\n".join(written.read_text().splitlines()[:12]))

['IEEE13_cdpsm.glm']

clock {
  timezone UTC0;
  starttime '2000-01-01 00:00:00';
  stoptime '2000-01-01 00:00:00';
}
#set relax_naming_rules=1
module powerflow {
  solver_method NR;
  nominal_frequency 60;
  line_capacitance TRUE;
}
module generators;


The records follow, each as `object <class> { ... }` with the name first:

In [11]:
text = written.read_text()
start = text.index('object load {')
print(text[start:text.index('}', start) + 1])

object load {
  name "ld_611";
  parent "611";
  phases "CN";
  nominal_voltage 2401.77711982884;
  base_power_C 187882.94;
  current_pf_C 0.904819;
  current_fraction_C 1;
}


The written `.glm` reads back through `GlmReader`, with the same counts for the classes that round-trip:

In [12]:
again = GlmReader(written).read_all()
print(again.count(schema.OverheadLine), again.count(schema.Load), again.count(schema.Transformer))

11 9 3


## Solve in GridLAB-D

The `solve` extra installs GridLAB-D 6.0 as the `gridlabd` wheel (`pip install --pre "cimhub-gridlabd[solve]"`), which runs
the engine from Python. The written `.glm` carries its own clock and solver, so it loads as it is. The wheel reads
GridLAB-D's polar text (`2401.78-120d`) as rectangular, so set `complex_output_format` to `RECT` before you read a
voltage. `load_glm` and `run` return 0 on success.

In [13]:
import cmath
import math

import gridlabd

engine = gridlabd.GridLabD()
engine.set_working_directory(str(written.parent))
assert engine.load_glm([written.name]) == 0
engine.global_setvar("complex_output_format", "RECT")
assert engine.run() == 0
for bus in ("650", "632", "671", "675"):
    volts = [engine.get_object_property_value(bus, f"voltage_{leg}") for leg in "ABC"]
    print(bus, "  ".join(f"{abs(v):7.1f} V {math.degrees(cmath.phase(v)):7.2f}°" for v in volts))

650  2401.7 V  -30.00°   2401.7 V -150.00°   2401.7 V   90.00°
632  2478.3 V  -32.44°   2471.3 V -151.04°   2481.4 V   88.13°
671  2410.4 V  -35.29°   2462.4 V -151.42°   2399.4 V   86.47°
675  2395.4 V  -35.53°   2467.6 V -151.59°   2395.2 V   86.47°


`cim_to_glm(network, path)` does the export and the write in one call, and `glm_to_cim` does the import. Chained:

```python
from cimhub_gridlabd.exporter.cim_to_glm import cim_to_glm
from cimhub_gridlabd.glm_reader import GlmReader
from cimhub_gridlabd.importer.glm_to_cimtbl import glm_to_cim

network = glm_to_cim(GlmReader(deck, defines={"VSOURCE": "66395.28"}).read_all(), name="IEEE13_cdpsm")
cim_to_glm(network, "out/IEEE13_cdpsm.glm", name_source="name")
```

## Next

- [Import](import.md): reader and importer options, the artifact files and what raises.
- [Export](export.md): names, the header, switch state and what raises.
- [Players and schedules](time-series.md): time series and the `time_points` tables.
- [Round trip](roundtrip.md): checking a converted model against the original in GridLAB-D.
- [What converts](coverage.md): the GLM classes that import and export.